# Tujya, step 2.2: detector `yolov8n` on C3PI v2

YOLOv8n (Ultralytics, one-stage, 3.2 M parameters). Branch `experiment/detector-yolov8n` of https://github.com/melekbouabid/syRX.

What this notebook does (code: `src/models/detection/runner.py`, search space:
`config/hpo/detector_yolov8n.yaml`):
1. Optuna search (seeded TPE): each trial trains the pretrained model briefly on part of the
   train split and is scored on the validation split (`val_mAP50_95`);
2. the best trial is retrained on the whole train split;
3. test scores on the comparison split (`test_v2_*`) and the whole v2 test (`test_v2_all_*`),
   ONNX export, CPU latency; everything is written to an export folder, zipped at the end.

Settings before running: **Accelerator** GPU T4 x1; **Internet** on (clone and `pip`);
**Add data**: your private dataset `tujya-c3pi-detect-v2` (`python -m src.models.detection.package_kaggle`).
Then **Save Version > Save & Run All (Commit)**: the run continues in the background
(about 3 h). Download `detector_yolov8n_export.zip` from the Output tab and import it:
`python -m src.models.detection.import_kaggle detector_yolov8n_export.zip`
(docs/module1/kaggle_detecteurs.md). No secret is needed: never put a token in a notebook.

In [ ]:
import os

ARCH = "yolov8n"
BRANCH = "experiment/detector-yolov8n"
REPO = "https://github.com/melekbouabid/syRX.git"
DATASET = "tujya-c3pi-detect-v2"  # slug of the private Kaggle dataset
# Smoke mode: tiny subset, 2 trials, 1 epoch (a check of the notebook, not an experiment).
SMOKE = os.environ.get("TUJYA_SMOKE", "0").lower() in ("1", "true", "yes")

In [ ]:
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO_DIR = WORK / "syRX"
    if not REPO_DIR.exists():
        clone = ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(REPO_DIR)]
        subprocess.run(clone, check=True)
    requirements = REPO_DIR / "notebooks" / "kaggle" / "requirements.txt"
    pip = [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
    subprocess.run(pip, check=True)
else:  # local smoke run, from the repository
    here = [Path.cwd(), *Path.cwd().parents]
    REPO_DIR = next(p for p in here if (p / "pyproject.toml").is_file())
    WORK = REPO_DIR / "runs" / "notebook"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"], capture_output=True, text=True
)
print("code:", commit.stdout.strip())

In [ ]:
import torch

if ON_KAGGLE:
    found = sorted(Path("/kaggle/input").rglob("test_holdout.txt"))
    assert found, f"Add the private dataset {DATASET} to this notebook (Add data)."
    DATA = found[0].parent
else:
    DATA = REPO_DIR / "data" / "processed" / "c3pi" / "datasets_v2" / "detect"
counts = {s: len(list((DATA / "images" / s).glob("*.jpg"))) for s in ("train", "val", "test")}
print(DATA, counts)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from src.models.detection.runner import run_experiment

OUT = WORK / (f"detector-{ARCH}-smoke" if SMOKE else f"detector-{ARCH}")
config = REPO_DIR / "config" / "hpo" / f"detector_{ARCH}.yaml"
summary = run_experiment(config, DATA, OUT, smoke=SMOKE)

In [ ]:
import pandas as pd

print(pd.read_csv(OUT / "trials.csv").to_string(index=False))
final = summary["final"]
print("best hyperparameters:", final["params"])
wanted = ("val_mAP", "test_v2_mAP", "test_v2_precision", "test_v2_recall", "test_v2_all_mAP")
print({k: round(v, 4) for k, v in final["metrics"].items() if k.startswith(wanted)})
print(final["latency"])

In [ ]:
import shutil

base = str(WORK / f"detector_{ARCH}_export")
archive = shutil.make_archive(base, "zip", OUT.parent, OUT.name)
print(archive, round(Path(archive).stat().st_size / 1e6, 1), "MB")

## Next steps (on the PC)

1. Download `detector_yolov8n_export.zip` (Output tab of the saved version).
2. `docker compose up -d mlflow`, then, in the training venv:
   `python -m src.models.detection.import_kaggle detector_yolov8n_export.zip`
   (parent run + one child run per trial in `tujya-vision`, Registry version,
   `promote_if_better` against the champion re-evaluated on the same split).
3. Weights: `dvc add` of the imported export folder (never git), see
   docs/module1/kaggle_detecteurs.md.